In [1]:
import os
import re
import numpy as np
from collections import Counter

In [2]:
# Here is a Word2Vec Implementation, 
# to learn the basic which would lead to transformers and other concepts of understanding an LLM from scratch

In [3]:
def preprocess(text):
    # Preprocess each text obtained with the following
    # 1. Lowercase conversion
    text = text.lower()
    # 2. Strip punctuation
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # 3. Tokenize Words
    tokens = text.split()
    return tokens


MIN_COUNT = 5


def preprocess(texts, min_count=MIN_COUNT):
    """Preprocess the whole corpus and build the vocabulary in one pass.

    1. Lowercase conversion
    2. Strip punctuation
    3. Tokenize Words
    4. Build vocabulary: count word frequencies, drop words with
       frequency < min_count (typically 5).
    5. word2idx
    6. idx2word

    Args:
        texts: iterable of raw text strings (one per document/file).
        min_count: drop words occurring fewer than this many times.

    Returns:
        tokenized_docs: list of token lists, one per input document.
        vocab: word -> frequency for words with count >= min_count.
        word2idx: word -> integer index.
        idx2word: integer index -> word.
    """
    tokenized_docs = []
    counter = Counter()
    for text in texts:
        # 1-3. Lowercase, strip punctuation, tokenize.
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        tokens = text.split()
        tokenized_docs.append(tokens)
        counter.update(tokens)

    # 4. Drop every word occurring fewer than min_count times across the whole
    # corpus. Retained words kept in (deterministic) insertion order.
    vocab = {word: count for word, count in counter.items() if count >= min_count}

    # 5. word2idx: map each retained word to a unique integer index.
    word2idx = {word: i for i, word in enumerate(vocab)}

    # 6. idx2word: reverse mapping from index back to word.
    idx2word = {i: word for word, i in word2idx.items()}

    return tokenized_docs, vocab, word2idx, idx2word


COMBINED_DIR = "data/combined"


def build_combined_corpus(corpora, output_dir=COMBINED_DIR):
    """Concatenate every text in each corpus directory into one file per author.

    Downstream steps then read a single file per author instead of reprocessing
    each book on every run. Output: <output_dir>/<author>.txt
    """
    os.makedirs(output_dir, exist_ok=True)

    for corpus_dir in corpora:
        author = os.path.basename(corpus_dir)  # e.g. "tolstoy"
        combined = []
        for filename in sorted(os.listdir(corpus_dir)):
            if not filename.endswith(".txt"):
                continue
            with open(os.path.join(corpus_dir, filename), "r", encoding="utf-8") as file:
                combined.append(file.read())

        output_path = os.path.join(output_dir, f"{author}.txt")
        with open(output_path, "w", encoding="utf-8") as file:
            file.write("\n\n".join(combined))
        print(f"Saved {output_path}")

In [4]:
if __name__ == "__main__":
    corpora = ["data/tolstoy", "data/dostoevsky"]

    # 7. Pass tolstoy as one file and dostoevsky as another file.
    build_combined_corpus(corpora)
    
    texts = [
        open(os.path.join(COMBINED_DIR, "tolstoy.txt"), encoding="utf-8").read(),
        open(os.path.join(COMBINED_DIR, "dostoevsky.txt"), encoding="utf-8").read(),
    ]

    tokenized_docs, vocab, word2idx, idx2word = preprocess(texts)

    total_tokens = sum(len(tokens) for tokens in tokenized_docs)

    print(f"Total tokens: {total_tokens}")
    print(f"Vocabulary size (min_count={MIN_COUNT}): {len(vocab)}")
    print("Example kept words:", list(vocab.items())[:10])
    print("word2idx sample:", {w: word2idx[w] for w in list(word2idx)[:5]})
    print("idx2word sample:", {i: idx2word[i] for i in range(5)})

    # Sanity: mappings are consistent and round-trip correctly.
    assert len(word2idx) == len(idx2word) == len(vocab)
    assert all(idx2word[word2idx[w]] == w for w in vocab)

Saved data/combined/tolstoy.txt
Saved data/combined/dostoevsky.txt
Total tokens: 2659120
Vocabulary size (min_count=5): 15839
Example kept words: [('anna', 1084), ('karenina', 35), ('by', 10339), ('leo', 10), ('tolstoy', 7), ('part', 1119), ('one', 10489), ('chapter', 1148), ('1', 69), ('happy', 715)]
word2idx sample: {'anna': 0, 'karenina': 1, 'by': 2, 'leo': 3, 'tolstoy': 4}
idx2word sample: {0: 'anna', 1: 'karenina', 2: 'by', 3: 'leo', 4: 'tolstoy'}


### Subsampling of Frequent Words
Applied *before* generating training windows, to thin out high-frequency, low-information words ("the", "is", "and").

For each word `w` with corpus frequency `f(w)` (fraction of total tokens):

$$ 
P_{discard}(w) = 1 - \sqrt{\frac{t}{f(w)}}
$$

- `t` ≈ 1e-5 (threshold, tunable).
- For each occurrence of `w` in the corpus, discard it with probability `P_discard(w)` before window generation.
- Words rarer than `t` get `P_discard ≈ 0` (never discarded); very frequent words get discarded often.

## 3. Negative Sampling Distribution
Precompute a sampling table over the vocabulary, weighted by:

$$
P(w) = \frac{f(w)^{3/4}}{\sum_{j=1}^{V} f(w_j)^{3/4}}
$$

- `f(w)` = raw frequency count (or unigram probability) of word `w`.
- The `3/4` power dampens dominance of very frequent words, boosts relative chance of rare words being sampled as negatives.
- Implementation trick: build a large array (e.g. size 1e8) where each word index appears proportional to `P(w)`, then sample by random indexing — O(1) per draw.

In [5]:
# Implementation of 2. and 3.
t = 1e-5
rng = np.random.default_rng(seed=42)

# 3. Negative sampling distribution over the FULL vocabulary, weighted
#    by f(w)^(3/4) and normalized so it sums to 1.
sum_f_w_34 = 0.0
for w in vocab:
    sum_f_w_34 += (vocab[w] / total_tokens) ** 0.75
p_w_neg_sam = {}
for w in vocab:
    f_w_34 = (vocab[w] / total_tokens) ** 0.75
    p_w_neg_sam[word2idx[w]] = f_w_34 / sum_f_w_34

# 2. Subsample token OCCURRENCES: discard each occurrence of w with
#    probability P_discard(w) = 1 - sqrt(t / f(w)). One shared RNG stream
#    (deterministic) is drawn for every token; words missing from the
#    vocabulary (freq < min_count) are dropped from the stream entirely.
pkeep = []
for doc in tokenized_docs:
    kept = []
    for w in doc:
        if w not in vocab:
            continue  # unknown word (freq < min_count)
        f_w = vocab[w] / total_tokens
        p_keep = min(1.0, (t / f_w) ** 0.5)
        if rng.uniform(0.0, 1.0) < p_keep:
            kept.append(w)
    pkeep.append(kept)

In [6]:
k = 5
pairs = []
for doc in pkeep:
    for i in range(len(doc)):
        center = doc[i]
        window_start = max(0, i - k)
        window_end = min(len(doc), i + k + 1)
        context_words = doc[window_start:i] + doc[i + 1 : window_end]
        for context in context_words:
            pairs.append((center, context))

In [7]:
len(pairs)

4942460